# 📡 Telco Customer Churn — Enhanced EDA

**Dataset:** IBM Telco Customer Churn ([Kaggle](https://www.kaggle.com/blastchar/telco-customer-churn))  
**Original Tutorial Inspiration:** [Channel Name — Link] *(Attribution: base EDA structure)*  

**Extensions & Improvements in this version:**
- Consistent professional colour scheme (`#4DA8DA` / `#E8665A`) across all charts  
- Donut chart for target variable instead of a basic bar  
- KDE-overlaid histograms for numerical features split by churn  
- Box plots for charge distributions  
- Churn rate by tenure group with dual-axis (rate + volume)  
- Horizontal churn-rate bar charts per category (with n-counts)  
- Signed correlation bar chart (positive vs negative correlation)  
- Triangular (lower-half) annotated heatmap  
- Stacked percentage bars for Contract and Payment Method  


## 1 · Imports & Global Style

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ── Colour palette ─────────────────────────────────────────────────────────────
CLR_NO  = '#4DA8DA'   # blue  → retained customers
CLR_YES = '#E8665A'   # coral → churned customers
PALETTE = [CLR_NO, CLR_YES]

# ── Global plot defaults ───────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', context='notebook', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi'       : 120,
    'figure.facecolor' : 'white',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.titlepad'    : 14,
    'axes.titlesize'   : 14,
    'axes.labelsize'   : 12,
    'legend.fontsize'  : 10,
})
print("Libraries loaded ✔")

## 2 · Load & Inspect Data

In [ ]:
telco_base_data = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f"Shape: {telco_base_data.shape[0]:,} rows × {telco_base_data.shape[1]} columns")
telco_base_data.head()

In [ ]:
# Dtypes & descriptive statistics
print("=== Data Types ===")
print(telco_base_data.dtypes.to_string())

In [ ]:
# Numeric summary
# Note: SeniorCitizen is really categorical (0/1), so its stats look odd here
telco_base_data.describe().round(2)

## 3 · Target Variable — Churn Distribution

The dataset is **imbalanced at ~73 : 27** (retained : churned).  
We keep this in mind when interpreting feature-level churn rates later.


In [ ]:
churn_counts = telco_base_data['Churn'].value_counts()
total        = churn_counts.sum()

fig, ax = plt.subplots(figsize=(7, 7))

wedges, _, autotexts = ax.pie(
    churn_counts.values,
    colors     = PALETTE,
    autopct    = '%1.1f%%',
    startangle = 90,
    wedgeprops = dict(width=0.60, edgecolor='white', linewidth=3),
    pctdistance= 0.80,
)
for at in autotexts:
    at.set(fontsize=15, fontweight='bold', color='white')

# Legend below the donut
ax.legend(wedges, [f'No Churn  ({churn_counts["No"]:,})',
                   f'Churn  ({churn_counts["Yes"]:,})'],
          loc='lower center', bbox_to_anchor=(0.5, -0.10),
          ncol=2, fontsize=12, frameon=False)

# Centre text
ax.text(0, 0, f"{total:,}\nCustomers",
        ha='center', va='center', fontsize=14, fontweight='bold', color='#333')

ax.set_title('Customer Churn Distribution', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 4 · Data Cleaning

Steps:
1. Copy the base data so the original stays untouched  
2. Convert `TotalCharges` to numeric (it's stored as a string in the raw CSV)  
3. Drop the 11 rows where `TotalCharges` is blank — only 0.15% of data, safe to remove  
4. Bin `tenure` into 12-month groups for cleaner segmentation  
5. Drop `customerID` (not a feature) and raw `tenure` (replaced by group)  


In [ ]:
telco_data = telco_base_data.copy()

# Fix TotalCharges dtype
telco_data['TotalCharges'] = pd.to_numeric(telco_data['TotalCharges'], errors='coerce')

# Identify & drop blank rows
blank_rows = telco_data[telco_data['TotalCharges'].isnull()]
print(f"Rows with blank TotalCharges: {len(blank_rows)}  ({100*len(blank_rows)/len(telco_data):.2f}%)")
telco_data.dropna(inplace=True)
print(f"Rows remaining: {len(telco_data):,}")

In [ ]:
# Tenure bins (12-month windows)
labels = [f"{i}–{i+11} mo" for i in range(1, 72, 12)]
telco_data['tenure_group'] = pd.cut(
    telco_data['tenure'], range(1, 80, 12), right=False, labels=labels
)

print("Tenure group counts:")
print(telco_data['tenure_group'].value_counts().sort_index().to_string())

In [ ]:
# Drop columns not needed for analysis
telco_data.drop(columns=['customerID', 'tenure'], inplace=True)
print(f"Final shape: {telco_data.shape}")
telco_data.head()

## 5 · Missing Values Check

In [ ]:
missing_pct = (telco_data.isnull().sum() / len(telco_data) * 100)
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=True)

if missing_pct.empty:
    print("✅  No missing values — dataset is clean.")
else:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(missing_pct.index, missing_pct.values, color=CLR_YES, alpha=0.85)
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Values by Feature', fontweight='bold')
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    plt.tight_layout()
    plt.show()

## 6 · Encode Variables

In [ ]:
# Encode target: Yes → 1, No → 0
telco_data['Churn'] = (telco_data['Churn'] == 'Yes').astype(int)

# One-hot encode all remaining categoricals
telco_dummies = pd.get_dummies(telco_data)
print(f"Shape after encoding: {telco_dummies.shape}")
telco_dummies.head(3)

## 7 · Numerical Feature Distributions

KDE-overlaid histograms make it easy to spot where churned vs retained customers cluster.


In [ ]:
num_cols = ['MonthlyCharges', 'TotalCharges']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Numerical Features — Distribution by Churn',
             fontsize=15, fontweight='bold', y=1.02)

for ax, col in zip(axes, num_cols):
    for churn_val, color, label in [(0, CLR_NO, 'No Churn'), (1, CLR_YES, 'Churn')]:
        subset = telco_dummies.loc[telco_dummies['Churn'] == churn_val, col]
        ax.hist(subset, bins=40, density=True, alpha=0.40, color=color, label=label)
        subset.plot.kde(ax=ax, color=color, linewidth=2.5)

    ax.set_title(col, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.legend()

plt.tight_layout()
plt.show()

### Box Plots — Charges vs Churn

Box plots reveal the median and spread differences more clearly than KDE for skewed distributions.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Charge Distribution by Churn', fontsize=15, fontweight='bold', y=1.02)

for ax, col in zip(axes, ['MonthlyCharges', 'TotalCharges']):
    sns.boxplot(data=telco_dummies, x='Churn', y=col,
                palette=PALETTE, width=0.5, linewidth=1.5, fliersize=3, ax=ax)
    ax.set_xticklabels(['No Churn', 'Churn'])
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

**Insights:**  
- **Monthly Charges**: Churned customers pay significantly more per month (median ~$75 vs ~$60)  
- **Total Charges**: Churned customers have lower total charges because they leave early — consistent with the tenure pattern we'll see next  


## 8 · Churn Rate by Tenure Group

Dual axis: bar = churn rate (%), line = number of customers in each group.


In [ ]:
tenure_stats = (
    telco_data
    .groupby('tenure_group', observed=True)['Churn']
    .agg(ChurnRate='mean', Customers='count')
    .reset_index()
)
tenure_stats['Churn %'] = tenure_stats['ChurnRate'] * 100
labels_x = tenure_stats['tenure_group'].astype(str).tolist()

fig, ax1 = plt.subplots(figsize=(11, 5))

bar_colors = [CLR_YES if r > 30 else CLR_NO for r in tenure_stats['Churn %']]
bars = ax1.bar(labels_x, tenure_stats['Churn %'],
               color=bar_colors, alpha=0.85, width=0.6, edgecolor='white', zorder=2)

ax1.set_ylabel('Churn Rate (%)', color='#444', fontsize=12)
ax1.set_xlabel('Tenure Group', fontsize=12)
ax1.set_title('Churn Rate by Tenure Group', fontsize=15, fontweight='bold')
ax1.set_ylim(0, 65)
ax1.yaxis.set_major_formatter(mtick.PercentFormatter())
ax1.tick_params(axis='y', labelcolor='#444')

for bar, val in zip(bars, tenure_stats['Churn %']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.0,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold', color='#333')

ax2 = ax1.twinx()
ax2.plot(labels_x, tenure_stats['Customers'],
         color='#555', marker='o', linewidth=2.2, markersize=7, zorder=3, label='# Customers')
ax2.set_ylabel('Number of Customers', color='#555', fontsize=12)
ax2.tick_params(axis='y', labelcolor='#555')
ax2.spines['right'].set_visible(True)

patches_leg = [
    mpatches.Patch(color=CLR_YES, label='Churn Rate > 30%'),
    mpatches.Patch(color=CLR_NO,  label='Churn Rate ≤ 30%'),
    plt.Line2D([0], [0], color='#555', marker='o', linewidth=2, label='# Customers')
]
ax1.legend(handles=patches_leg, loc='upper right', fontsize=10, frameon=True)

plt.tight_layout()
plt.show()

**Key Insight:** Churn is highest in the first year (~47%) and drops dramatically for customers who stay beyond 2 years. Retention efforts should focus on the **1–12 month** window.


## 9 · Churn Rate by Categorical Feature

Horizontal bar charts sorted by churn rate, with customer counts annotated. Bars above 30% shown in coral.


In [ ]:
cat_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
            'PhoneService', 'InternetService', 'Contract', 'PaymentMethod',
            'OnlineSecurity', 'TechSupport', 'tenure_group']

n_cols = 2
n_rows = (len(cat_cols) + 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4.5))
fig.suptitle('Churn Rate by Feature Category', fontsize=17, fontweight='bold', y=1.01)
axes = axes.flatten()

for ax, col in zip(axes, cat_cols):
    grp = (telco_data
           .groupby(col, observed=True)['Churn']
           .agg(ChurnRate='mean', N='count')
           .reset_index()
           .sort_values('ChurnRate', ascending=True))
    grp['Churn %'] = grp['ChurnRate'] * 100

    colors = [CLR_YES if v > 30 else CLR_NO for v in grp['Churn %']]
    bars = ax.barh(grp[col].astype(str), grp['Churn %'],
                   color=colors, alpha=0.85, edgecolor='white')

    ax.set_xlabel('Churn Rate (%)')
    ax.set_title(col, fontweight='bold')
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())

    for bar, (_, row) in zip(bars, grp.iterrows()):
        ax.text(bar.get_width() + 0.5,
                bar.get_y() + bar.get_height() / 2,
                f"{row['Churn %']:.1f}%  (n={row['N']:,})",
                va='center', fontsize=9)
    ax.set_xlim(0, grp['Churn %'].max() * 1.45)

# Hide any unused axes
for ax in axes[len(cat_cols):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

## 10 · Contract Type & Payment Method — Stacked Percentage Bars

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Retained vs Churned (%) by Segment', fontsize=15, fontweight='bold')

for ax, col in zip(axes, ['Contract', 'PaymentMethod']):
    ct = (telco_data.groupby([col, 'Churn'], observed=True)
          .size().unstack(fill_value=0))
    pct = ct.div(ct.sum(axis=1), axis=0) * 100

    pct.plot(kind='bar', stacked=True, color=PALETTE,
             ax=ax, width=0.6, edgecolor='white', legend=False)

    ax.set_ylabel('Customers (%)')
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')

    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f%%', label_type='center',
                     fontsize=10, fontweight='bold', color='white')

handles = [mpatches.Patch(color=CLR_NO, label='No Churn'),
           mpatches.Patch(color=CLR_YES, label='Churn')]
fig.legend(handles=handles, loc='upper right', fontsize=11, frameon=False)

plt.tight_layout()
plt.show()

## 11 · Correlation Analysis

### 11a · Feature Correlation with Churn (Signed Bar Chart)

Bars to the **right** (coral) = higher churn associated with this feature  
Bars to the **left** (blue) = lower churn (protective factor)


In [ ]:
churn_corr = (telco_dummies.corr()['Churn']
              .drop('Churn')
              .sort_values())

colors = [CLR_YES if c > 0 else CLR_NO for c in churn_corr.values]

fig, ax = plt.subplots(figsize=(10, 14))
ax.barh(churn_corr.index, churn_corr.values, color=colors, alpha=0.85, edgecolor='white')
ax.axvline(0, color='#333', linewidth=1.2, linestyle='--', zorder=3)
ax.set_xlabel('Pearson Correlation with Churn', fontsize=12)
ax.set_title('Feature Correlation with Customer Churn', fontsize=15, fontweight='bold')

patches_leg = [
    mpatches.Patch(color=CLR_YES, label='Positive → Higher Churn Risk'),
    mpatches.Patch(color=CLR_NO,  label='Negative → Lower Churn Risk'),
]
ax.legend(handles=patches_leg, loc='lower right', fontsize=11)
plt.tight_layout()
plt.show()

### 11b · Full Feature Correlation Heatmap (Lower Triangle Only)

In [ ]:
corr_matrix = telco_dummies.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))   # hide redundant upper half

fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(
    corr_matrix, mask=mask, cmap='RdYlBu_r', center=0,
    annot=False, linewidths=0.3, linecolor='white',
    cbar_kws={'shrink': 0.65, 'label': 'Pearson r'},
    ax=ax
)
ax.set_title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=16)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## 12 · Key Findings & Business Recommendations

### 📊 What drives churn?

| Factor | Churn Rate | Insight |
|---|---|---|
| Month-to-month contract | ~43% | Highest churn — no lock-in |
| No online security | ~42% | Feel under-protected |
| No tech support | ~42% | Poor service experience |
| Fibre optic internet | ~42% | High cost, high expectations |
| Electronic check payment | ~45% | Usually month-to-month customers |
| Tenure < 12 months | ~47% | First-year is the critical window |

### 🔒 What protects against churn?

- **Two-year contracts** → only ~3% churn
- **Tenure > 5 years** → churn rate drops below 10%  
- **Having dependents or a partner** → slightly more stable  
- **Automatic payment methods** (bank transfer / credit card) → lower churn

### 💡 Business Recommendations

1. **Offer discounts or perks** to convert month-to-month customers to annual/two-year plans  
2. **Proactive outreach in months 1–6** — this is when most churn happens  
3. **Bundle online security + tech support** into onboarding packages  
4. **Review Fibre Optic pricing** — high monthly charges are strongly correlated with churn  
5. **Incentivise auto-pay** — customers paying by electronic check churn at nearly 2× the rate  


## 13 · Export Processed Data

In [ ]:
telco_dummies.to_csv('telco_churn_processed.csv', index=False)
print(f"Exported {len(telco_dummies):,} rows × {telco_dummies.shape[1]} columns")